# Learning objectives

- Validate a public MCP endpoint.
- Create an MCP-enabled agent using the SDK.
- Inspect MCP tool calls and approval steps.

# Configure the MCP tool

The example below constructs an MCPTool configured for the selected public endpoint.   
Note: require_approval is set to "never" only because the selected server's known tools are read-only and never silently bypass a requested approval step.

In [ ]:
# Import local configuration helpers, the Foundry SDK, and Azure credential support.
import os

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from workshop_core.config import WorkshopConfig
from workshop_core.naming import build_resource_name, sdk_participant_id

# Load workshop configuration.
load_dotenv()
config = WorkshopConfig.load()
sdk_participant = sdk_participant_id(config.participant_id)
agent_name = build_resource_name(sdk_participant, "mcp", "agent")

# Define a prompt which is relevant to the MCP tool defined.
# Notice we're not being too specific about the tool usage, we expect the agent to reason about the tool usage based on the prompt and the tool definition.
prompt = (
    "Find current Microsoft Foundry guidance "
    "for remote MCP tools."
)
MCP_ENDPOINT = "https://learn.microsoft.com/api/mcp"

# Authenticate with the Azure identity established by az login and connect to Foundry.
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai = project.get_openai_client()

# Declare the approved read-only Microsoft Learn MCP server for the agent definition.
mcp_tool = MCPTool(
    server_label="microsoft-learn",
    server_url=MCP_ENDPOINT,
    require_approval="never",
)

# Create the MCP-enabled agent and verify tool use

The cell below creates an agent definition that retains the Lab 1 support-assistant safety instructions, requires the MCP tool for current Foundry documentation questions, then invokes the model and asserts the metadata shows an MCP tool call

In [ ]:
# Define the support-agent instructions and direct current Foundry questions to MCP.
instructions = (
    "You are the Contoso SupportHub X1 support assistant. "
    "Answer only from connected sources when they are available. "
    "If a source does not contain the answer, say so. "
    "Never invent customer or private information. "
)

# Create an agent version that includes the configured MCP tool.
mcp_agent = project.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=config.model_deployment_name,
        instructions=instructions,
        tools=[mcp_tool],
    ),
)

# Create a managed conversation and send the documentation question to the agent.
mcp_conversation = openai.conversations.create()
mcp_response = openai.responses.create(
    conversation=mcp_conversation.id,
    input=prompt,
    extra_body={
        "agent_reference": {
            "name": mcp_agent.name,
            "type": "agent_reference",
        }
    },
)

# Confirm that the response contains at least one MCP tool-call record.
mcp_calls = [item for item in mcp_response.output if item.type == "mcp_call"]
assert mcp_calls, f"Expected the Microsoft Learn MCP tool to be invoked, but no tool calls were found in the response.\nTry being more specific in your prompt to use the MCP tool."
print("MCP tool call successfully invoked.\n")
print(f"{mcp_response.output_text[:500]}............. truncated for brevity")

# Inspection

Loop over MCP tool-call output items and print labeled fields.   
Do not auto-approve; if approval is required the notebook will present the approval step to the operator.

In [ ]:
# Inspect the MCP call records that prove the tool was invoked by the runtime.
for item in mcp_calls:
    server_label = getattr(item, "server_label", None)
    if server_label is None:
        raise RuntimeError("MCP call did not include a server label for inspection.")

    # Print the tool metadata available from the current response contract.
    print("\nMCP server label:", server_label)
    print("Tool name:", getattr(item, "tool_name", "<unknown>"))
    print("Tool-call arguments:", getattr(item, "arguments", "<no arguments>"))
    print("Tool-call status:", getattr(item, "status", "<no status>"))

# Do not auto-approve tool actions; the operator must follow any approval step presented.